## Running VGAE / Graph-PCA & baseline models

- Dim-reduction on features ($x \in \mathbb{R}^{N \times P}$: observation $z \in \mathbb{R}^{N \times D}$: representation, $u \in \mathbb{R}^{N \times 1}$: interpretability as "trajectory / gradient")
- Reconstruction w/ $\sigma(z \cdot z^T)$ for graph reconstruction, regularize $u$ w/ CyCIF graph prior
- Benchmark against K-means clustering & PCA, regular VAE, etc.

In [ ]:
import os
import gc
import sys
import pickle
import gzip

import numpy as np
import cupy as cp
import pandas as pd
import scanpy as sc

import torch
import tifffile
import torch.nn as nn

import networkx as nx
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from torch_geometric import utils as pyg_utils


In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, gen_graph, configs, dataset, zonation
import baseline, sb_vgae, model_train, model_eval

from torch_geometric.loader import DataLoader

### VGAE (Xenium-DESI integration)

- Integrate DESI prior to guide latent prob. clustering of Xenium

In [ ]:
import json
import squidpy as sq

from sklearn.decomposition import FastICA
from util import IO, utils, gen_graph
from models import baseline, dataset
from torch_geometric import utils as pyg_utils

from importlib import reload
reload(IO)
reload(utils)
reload(baseline)
reload(gen_graph)
reload(dataset)


In [ ]:
# Load paired Xenium & DESI

xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
sample_id = 'NIH_F5'
# sample_ids = sorted([f for f in os.listdir(xenium_path) if os.path.isdir(os.path.join(xenium_path, f))])

# Load xenium
adata = IO.load_xenium(os.path.join(xenium_path, sample_id))
with open(os.path.join('../data/xenium/', sample_id, 'experiment.xenium'), 'r') as ifile:
    scalefactor = json.load(ifile)['pixel_size']
xenium_coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 


In [ ]:
# Load DESI pixels mapped to Xenium
ratio = 1/10  # Xenium -> DESI downscale ratio
adata_desi = IO.load_paired_desi(os.path.join(desi_path, sample_id+'.ome.tif'),
                                 adata_other=adata,
                                 ratio=ratio / scalefactor,
                                 dilate=True)
desi_coords = adata_desi.obs[['y_centroid', 'x_centroid']].copy().to_numpy()
                                 

In [ ]:
# Compute aux. DESI expressions `u`
# Experiment: Encoding w/ PCA or ICA

def get_PCs(adata, coords, 
            n_pcs, k=8, 
            graph_regularize=False):
    """
    Dimension reduction w/ (graph-regularized) PCA
    """
    if graph_regularize:
        G = gen_graph.construct_graph(coords, k=k, weighted=False)  
        edge_index = pyg_utils.from_networkx(G).edge_index
        model = baseline.GPCALayer(c_in=adata.X.shape[-1],
                                   c_out=n_pcs,
                                   center=True,
                                   init_weight=True,
                                   ortho_weight=True)
        U_gpca = model(torch.tensor(adata.X).float(), edge_index)
        adata.obsm['X_pca'] = U_gpca.detach().cpu().numpy()
    else:
        sc.pp.pca(adata, n_pcs)
    return None

n_aux = 32
get_PCs(adata_desi, desi_coords, 
        n_pcs=n_aux,
        graph_regularize=False)

# sc.pp.neighbors(adata_desi, n_neighbors=30, use_rep='X_pca')
# sc.tl.leiden(adata_desi, resolution=0.1)
# print(adata_desi.obs.leiden.value_counts())
# sc.pl.spatial(adata_desi, color='leiden', size=5)

# Append aux. DESI expression (PC) to Xenium
# adata.obsm['X_aux'] = adata_desi.obsm['X_pca']  

# n_aux = 32
# ica_model = FastICA(n_components=n_aux, random_state=0, whiten='unit-variance')
# s = ica_model.fit_transform(adata_desi.X)
# adata_desi.obsm['X_ica'] = s

adata.obsm['X_u'] = adata_desi.X
adata.obsm['X_aux'] = adata_desi.obsm['X_pca']

#### Model sketch iVAE: 
**TODO: simplicial vs. real-value latent (z)**
- $z_i \mid u_i \sim \mathcal{Dir}(f_{\lambda}(u_i))$ vs. $z_i \mid u_i \sim \mathcal{vMF}(f_{\lambda}(u_i))$
- $x_i \mid z_i, \mathbf{A} \sim \mathcal{NegBinom}(l \cdot f_{\theta}(z_i, \mathbf{A}, \theta_g)$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import pyro
import pyro.poutine as poutine
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, TraceEnum_ELBO
from pyro.optim import Adam

from torch_geometric.data import DataLoader

from models import logit_vgae, model_train

In [ ]:
xenium_loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=8) 
xenium_data = xenium_loader.load_graphs([adata])
xenium_train_dl = DataLoader(xenium_data, shuffle=True)

Observation: Linear ICA demix DESI well

In [ ]:
# from sklearn.decomposition import FastICA

# ica_model = FastICA(n_components=16, random_state=0, whiten='unit-variance')
# s_desi = ica_model.fit_transform(adata_desi.X)
# for i in range(s_desi.shape[1]):
#     adata_desi.obs['IC'+str(i)] = s_desi[:, i]

# ic_labels = ['IC'+str(i) for i in range(s_desi.shape[1])]
# sq.pl.spatial_scatter(adata_desi, color=ic_labels, cmap='coolwarm', size=20, img=False)

# del ica_model, s_desi, ic_labels

In [ ]:
from importlib import reload

reload(IO)
reload(utils)
reload(plot)
reload(dataset)
reload(configs)
reload(logit_vgae)
reload(model_train)

torch.manual_seed(0)

In [ ]:
lr = 1e-2
n_epochs = 500
device = torch.device('cuda')
n_aux = adata.obsm['X_aux'].shape[1]

train_configs = configs.set_train_configs(n_epochs=n_epochs, 
                                          lr=lr,
                                          gamma=1.,
                                          annealing=True,
                                          device=device)

model_configs = configs.set_model_configs(device=device,
                                          beta=0.5,
                                          c_in=adata.shape[1],
                                          c_u=adata_desi.shape[1],  # Raw auxiliary dim.
                                          c_aux=n_aux,              # Reduced auxiliary dim.
                                          c_hidden=n_aux,
                                          c_latent=8,
                                          k_hop=3,
                                          dropout=0.5) 

In [ ]:
# DEBUG: test Projected Normal / Von mises-Fisher?

pyro.clear_param_store()
model = logit_vgae.LogitVGAE(model_configs, device=train_configs.device)
model, losses = model_train.train_logit_vgae(model, xenium_train_dl, train_configs)

In [ ]:
plt.figure(figsize=(5, 2))
plt.plot(np.arange(train_configs.n_epochs), losses)
plt.xlabel('Epochs')
plt.ylabel('Train ELBO')
plt.show()

In [ ]:
# Inference on full data (CPU)

G_xenium = gen_graph.construct_graph(xenium_coords, 
                                     k=30, 
                                     r=np.inf)

edge_index = pyg_utils.from_networkx(G_xenium).edge_index

model.device = 'cpu'
model = model.to('cpu')

qz = model.get_z(adata.X.A, 
                 adata.obsm['X_u'], 
                 edge_index)

pz = model.get_cond_prior(adata.obsm['X_aux'],
                          edge_index)

px = model.get_x(adata.X.A, 
                 edge_index, 
                 qz)

qz = qz.detach().cpu().numpy()
pz = pz.detach().cpu().numpy()
px = px.detach().cpu().numpy()

del G_xenium
gc.collect()
torch.cuda.empty_cache()

In [ ]:
rand_indices = np.random.choice(np.arange(adata.shape[1]), size=200, replace=False)

plt.figure(figsize=(6, 4))
plt.scatter(adata.X.A[:, rand_indices].flatten(), px[:, rand_indices].flatten(), s=1)
plt.xlabel('X')
plt.ylabel("X_reconst")
plt.show()

del rand_indices

In [ ]:
sns.clustermap(np.corrcoef(pz.T), cmap='coolwarm')

In [ ]:
sns.clustermap(np.corrcoef(qz.T), cmap='coolwarm')

In [ ]:
# Check posterior collapse?
plot.disp_spatial_latents(adata, pz, ncols=4)

In [ ]:
plot.disp_spatial_latents(adata, qz, ncols=4)

#### Trajectory Inference

- Try both Xenium & mapped DESI

In [ ]:
# Load saved latent representations
adata_qz = sc.read_h5ad('../data/xenium/adata_NIH_F5_qz.h5ad')
qz = adata_qz.X

In [ ]:
adata_qz = sc.AnnData(qz)
sc.pp.pca(adata_qz)
sc.pp.neighbors(adata_qz, n_neighbors=30)
sc.tl.umap(adata_qz, n_components=3)

In [ ]:
# adata_qz = sc.AnnData(qz)
# sc.pp.pca(adata_qz)
# sc.pp.neighbors(adata_qz, n_neighbors=30)
# sc.tl.umap(adata_qz, n_components=3)

adata_norm = adata.copy()
sc.pp.normalize_total(adata_norm)
sc.pp.log1p(adata_norm)
sq.gr.spatial_neighbors(adata_norm, coord_type='generic', spatial_key='spatial')  # Spatial corr.
sq.gr.spatial_autocorr(adata_norm, mode='moran')  
# sc.pp.pca(adata_norm)


# # Assign learnt latent embeddings to Xenium / DESI dataset as metadata
adata_norm.obsm['X_pca'] = adata_qz.obsm['X_pca']
adata_norm.obsm['X_umap'] = adata_qz.obsm['X_umap']
adata_norm.obsm['X_z'] = qz

adata_desi.obsm['X_pca'] = adata_qz.obsm['X_pca']
adata_desi.obsm['X_umap'] = adata_qz.obsm['X_umap']
adata_desi.obsm['X_z'] = qz


In [ ]:
# PC on raw Xenium expressions

# sc.pp.pca(adata_norm)
# sc.pl.pca(adata_norm,
#           color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
#                  'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
#                  'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
#                 ],
#           size=10, 
#           cmap='RdBu_r')


In [ ]:
# Xenium markers
sc.pl.pca(adata_norm,
          color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
                 'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
                 'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
                ],
          size=20, 
          cmap='RdBu_r')


In [ ]:
sc.pl.umap(adata_norm,
          color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
                 'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
                 'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
                ],
          size=20, 
          cmap='RdBu_r')

In [ ]:
sc.pl.umap(adata_norm,
          color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
                 'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
                 'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
                ],
          size=20, 
          projection='3d', alpha=0.5,
          cmap='RdBu_r')

In [ ]:
# DESI markers
sc.pl.pca(adata_desi,
          color=['FA 20:4;O (C296)',
                 'FA 18:1; O (C286)',
                 'FA 22:5;O (C451)',
                 '808.6024 m/z ± 30 ppm (C40)',
                 '865.50838 m/z ± 50 ppm (C385)',
                 '673.49987 m/z ± 30 ppm (C10)',
                 '631.4937 m/z ± 30 ppm (C7)',
                 '346.05909 m/z ± 40 ppm (C126)',
                 '754.5545 m/z ± 30 ppm (C18)',
                 '258.11631 m/z ± 30 ppm (C1)',
                 '734.58637 m/z ± 30 ppm (C16)',
                 '703.58429 m/z ± 30 ppm (C13)',
                ],
          size=20, 
          cmap='RdBu_r')

In [ ]:
sc.pl.umap(adata_desi,
          color=['FA 20:4;O (C296)',
                 'FA 18:1; O (C286)',
                 'FA 22:5;O (C451)',
                 '808.6024 m/z ± 30 ppm (C40)',
                 '865.50838 m/z ± 50 ppm (C385)',
                 '673.49987 m/z ± 30 ppm (C10)',
                 '631.4937 m/z ± 30 ppm (C7)',
                 '346.05909 m/z ± 40 ppm (C126)',
                 '754.5545 m/z ± 30 ppm (C18)',
                 '258.11631 m/z ± 30 ppm (C1)',
                 '734.58637 m/z ± 30 ppm (C16)',
                 '703.58429 m/z ± 30 ppm (C13)',
                ],
          size=20, 
          cmap='RdBu_r')

In [ ]:
sc.pl.umap(adata_desi,
          color=['FA 20:4;O (C296)',
                 'FA 18:1; O (C286)',
                 'FA 22:5;O (C451)',
                 '808.6024 m/z ± 30 ppm (C40)',
                 '865.50838 m/z ± 50 ppm (C385)',
                 '673.49987 m/z ± 30 ppm (C10)',
                 '631.4937 m/z ± 30 ppm (C7)',
                 '346.05909 m/z ± 40 ppm (C126)',
                 '754.5545 m/z ± 30 ppm (C18)',
                 '258.11631 m/z ± 30 ppm (C1)',
                 '734.58637 m/z ± 30 ppm (C16)',
                 '703.58429 m/z ± 30 ppm (C13)',
                ],
          size=20,
          projection='3d', alpha=0.5,
          cmap='RdBu_r')

In [ ]:
adata_qz.write_h5ad('../data/xenium/adata_NIH_F5_qz.h5ad')

---

**Pseudotime sketch:**
- Fit an Elastic principal curve on `Z`
- Compute linear path btw principal "nodes"
- Compute k-NN distance from each cell to principal nodes --> final pseudotime assignment  

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import scFates as scf
import igraph

In [ ]:
n_pcurve_nodes = 30
scf.tl.curve(adata_qz, 
             Nodes=n_pcurve_nodes, 
             epg_mu=0.5, 
             epg_lambda=0.01,
             epg_trimmingradius=0.5,
             epg_extend_leaves=True,
             ndims_rep=adata_qz.shape[1])

In [ ]:
scf.pl.graph(adata_qz, basis='umap', nodes=list(np.arange(n_pcurve_nodes)))

`adata.uns['graph']['F']`: K x dim(Nodes) low-dim representation of the principal points <br>
`adata.uns['graph']['B']`: dim(Nodes) x dim(Nodes) adjacency matrix of the principal points <br>
`adata.obsm['X_R']`: soft assignment of cells to principal points

In [ ]:
from scanpy.tools._dpt import DPT
from scipy.spatial.distance import cdist

def get_pcurve_path(adata):
    """Compute linear trajectory btw principal nodes in latent space"""
    assert 'graph' in adata.uns.keys(), "Please run Principal Curve first"

    al = np.array(
        igraph.Graph.Adjacency(
            (adata.uns['graph']['B'] > 0).tolist(), 
            mode='undirected'
        ).get_edgelist()
    )

    root_node, term_node = adata.uns['graph']['tips']
    curr_node = root_node
    ypos, xpos = np.asarray(np.where(al == root_node)).T[0]  # coords of current node
    
    path = [curr_node]
    while term_node not in path:
        xpos = 1 - xpos  # adjacenct principle node
        curr_node = al[ypos, xpos]
        path.append(curr_node)
        if curr_node == term_node:
            break
        
        # Update `ypos`, `xpos` of `curr_node`:
        # besides root & term node, every other node appears twice 
        coords = np.asarray(np.where(al == curr_node)).T  # dim: [2, 2]
        if np.array_equal(coords[0], [ypos, xpos]):
            ypos, xpos = coords[1]
        else:
            ypos, xpos = coords[0]
        
    return path


def get_diffusion_dist(adata, root_expr, n_neighbors=50):
    # Append "dummy" principal node
    adata_root = sc.AnnData(np.expand_dims(root_expr, 0))
    adata_cat = sc.concat((adata, adata_root), axis=0)

    sc.pp.neighbors(adata_cat, n_neighbors=n_neighbors)
    
    dpt = DPT(adata_cat)
    dpt.compute_transitions()
    dpt.compute_eigen(n_comps=adata_cat.shape[1])
    dpt.iroot = adata_cat.shape[0] - 1
    dpt._set_pseudotime()
    
    return dpt.pseudotime[:-1]  # Drop "dummy" principal node


def get_geodesic_dist(pt1, pt2):
    pt1 = torch.tensor(pt1).float()
    pt2 = torch.tensor(pt2).float()

    u = pt1 / pt1.norm(dim=-1, keepdim=True)
    v = pt2 / pt2.norm(dim=-1, keepdim=True)

    dot_product = (u @ v).clamp(-1.0, 1.0)
    return torch.acos(dot_product).item()


def dist_to_pcurve(adata, n_neighbors=50, verbose=False):
    """
    Compute diffusion distance (DPT: D(x, y)) btw each cell (x) 
    and latent representation vector of each principal node (y)
    """ 
    assert 'graph' in adata.uns.keys(), "Please run Principal Curve first"
    
    pcurve_reprs = adata.uns['graph']['F'].T  # dim:[n_nodes, n_latent (K)]
    n_pts = adata.shape[0]
    n_nodes, n_latent = pcurve_reprs.shape
    dists = np.zeros((n_pts, n_nodes), dtype=np.float32)

    # Compute trajectory ordering of principal nodes
    pcurve_indices = get_pcurve_path(adata)
    if verbose:
        print('Principal Node ordering:', pcurve_indices)
    pcurve_reprs = pcurve_reprs[pcurve_indices, :]

    for i, pcurve in enumerate(pcurve_reprs):
        # dists[:, i] = cdist(adata.X, np.expand_dims(pcurve, 0)).squeeze()  # Euclidean distance
        dists[:, i] = np.array([get_geodesic_dist(z, pcurve) 
                                for z in adata.X])
        # dists[:, i] = get_diffusion_dist(adata, pcurve_repr[i], n_neighbors)  # Diffusion distance
    
    return dists


def compute_trajectory(adata, distances):
    """
    Compute smooth trajectory \in [0, 1] based on 
    distance to the sorted principal node
    """
    assert adata.shape[0] == distances.shape[0], \
        "Expression (`adata`) and distance assignment (`distances`) should have the same # samples"
    n_cells, n_nodes = distances.shape
    t = np.ones(n_cells, dtype=np.float32)
    t_values = np.arange(distances.shape[0]) / distances.shape[0]
    idx_t0 = 0

    for i in range(n_nodes):
        # Subset cells w/ the closest distance to the current principal nodes
        indices = np.where(distances.argmin(1) == i)[0]
        dist = distances[indices]  
        idx_tn = idx_t0 + dist.shape[0]

        if i == 0:
            sorted_indices = indices[dist[:, i].argsort()]
        elif i == n_nodes-1:
            sorted_indices = indices[dist[:, i].argsort()[::-1]]
        else:
            sorted_indices = indices[(dist[:, i-1]/dist[:, i+1]).argsort()]
            
        t[sorted_indices] = t_values[idx_t0:idx_tn]
        idx_t0 += dist.shape[0]

    adata.obs['t'] = t
    adata.obs['t_discrete'] = dists.argmin(1)  # Discrete 'pseudotime' assignment
    return None
    

In [ ]:
dists = dist_to_pcurve(adata_qz, verbose=True)
compute_trajectory(adata_norm, dists)

# labels = ['t'+str(i) for i in range(t.shape[1])]
# for i, label in enumerate(labels):
#     adata_norm.obs[label] = dist_to_pcurves[:, i]  # Assign DPT to each node on full data
# del i, label, labels

g = sns.heatmap(np.corrcoef(dists.T), cmap='coolwarm')
ax = g.axes
ax.axis('off')
ax.set_title('Distance correlations to\n Principal Nodes')

sc.pl.umap(adata_norm, color='t', cmap='RdBu', title='Pseudotime')

sq.pl.spatial_scatter(adata_norm, color='t', 
                      cmap='RdBu', size=20, img=False,
                      title='Pseudotime')

sq.pl.spatial_scatter(adata_norm, color='t_discrete', 
                      cmap='tab20', size=20, img=False,
                      title='Pseudotime\n (discrete)')

**TI w/ scFates:**

In [ ]:
import scFates as scf

In [ ]:
# adata_qz = sc.read_h5ad('../data/xenium/adata_NIH_F5_qz.h5ad')

# adata_norm.obsm['X_z'] = adata_qz.obs.values  # Sorted qz
# adata_norm.obsm['X_pca'] = adata_qz.obsm['X_pca']

# adata_desi.obsm['X_z'] = adata_qz.obs.values
# adata_desi.obsm['X_pca'] = adata_qz.obsm['X_pca']

In [ ]:
adata_norm.obsm['X_z'] = qz_sorted
scf.tl.curve(adata_norm, Nodes=30, use_rep='X_z', ndims_rep=adata_norm.obsm['X_z'].shape[-1])

In [ ]:
# TI w/ root assignment
scf.tl.root(adata_norm, "MYH11")

In [ ]:
scf.tl.pseudotime(adata_norm, n_jobs=16, n_map=100, seed=0)

In [ ]:
sc.pl.pca(adata_norm, color="t", cmap="coolwarm", title="Pseudotime\n (principal curve)")

In [ ]:
# Spatial distribution of trajectories
sq.pl.spatial_scatter(adata_norm, color='t', size=20, img=False, cmap='coolwarm', title='Spatial Trajectory\n (principal curve)')

DEGs along trajectory:

In [ ]:
# Xenium
scf.tl.test_association(adata_norm, n_jobs=16)
scf.tl.test_association(adata_norm, reapply_filters=True,A_cut=.5)
scf.pl.test_association(adata_norm)
ti_sig_features = adata_norm.var[adata_norm.var.signi].index

In [ ]:
scf.tl.fit(adata_norm, n_jobs=16)

Held-out validation:

In [ ]:
# adata_val = load_xenium(os.path.join(xenium_path, 'NIH_F4'))
# xenium_coords = adata_val.obs[['y_centroid', 'x_centroid']].copy().to_numpy()

# TODO: need to load paired `u`

G_xenium = gen_graph.construct_graph(xenium_coords, k=30, r=np.inf)
edge_index = pyg_utils.from_networkx(G_xenium).edge_index
qz, qz_Sigma = model.get_z(adata_val.X.A, edge_index)

del G_xenium, edge_index
gc.collect()

In [ ]:
plot.disp_spatial_latents(adata_val, qz, ncols=4)

Baselines: EM clustering:

In [ ]:
from sklearn.mixture import GaussianMixture

gm = GaussianMixture(n_components=10, random_state=0).fit(adata_norm.X.A)
gm_cov = gm.covariances_
gm_soft_assign = gm.predict_proba(adata_norm.X.A)

plot.disp_spatial_latents(adata, gm_soft_assign, ncols=2)

---